🎯 Scenario: The Field Marketing Lead has asked you to help prepare their team for conference season. Same Conference Prep Agent as DocumentSearchAgent.ipynb — web search, Wikipedia, internal notes — but instead of RAG (chunk/embed/retrieve), the agent maintains its own wiki of interlinked markdown pages. It reads the team's internal notes once, decides what concepts deserve their own page, writes them, and links them. Later it answers questions by reading the wiki it built.

Inspired by Andrej Karpathy's LLM Wiki idea: the agent manages its own memory.

## Install BeeAI Framework and set up our environment.

In [ ]:
%pip install -Uqq arize-phoenix arize-phoenix-otel s3fs fsspec==2025.3.0 \
 gcsfs nest_asyncio openinference-instrumentation-beeai jedi \
 beeai-framework beeai-framework[duckduckgo] beeai-framework[wikipedia] \
 langchain langchain_community wikipedia requests==2.32.4

!git clone https://github.com/sandijean90/beeai_framework_synthetic_data.git

from IPython.display import HTML, display
def set_css():
    display(HTML("\n<style>\n pre{\n white-space: pre-wrap;\n}\n</style>\n"))
get_ipython().events.register("pre_run_cell", set_css)

Import modules:

In [ ]:
import os
import re
import json
from datetime import date

import phoenix as px
from phoenix.otel import register
from dotenv import load_dotenv

from beeai_framework.agents.experimental import RequirementAgent
from beeai_framework.agents.experimental.requirements.conditional import ConditionalRequirement
from beeai_framework.backend import ChatModel, ChatModelParameters
from beeai_framework.memory import SummarizeMemory
from beeai_framework.middleware.trajectory import GlobalTrajectoryMiddleware
from beeai_framework.tools import Tool, tool
from beeai_framework.tools.search.duckduckgo import DuckDuckGoSearchTool
from beeai_framework.tools.search.wikipedia import WikipediaTool, WikipediaToolInput
from beeai_framework.tools.think import ThinkTool
from openinference.instrumentation.beeai import BeeAIInstrumentor

### Choose LLM (Ollama)

In [ ]:
provider = "ollama"
model = "granite3.3"

if provider == "ollama":
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    !nohup ollama serve >/dev/null 2>&1 &
    !ollama pull $model

llm = ChatModel.from_name(f"{provider}:{model}", ChatModelParameters(temperature=1))

### System Prompt

In [ ]:
instruct_prompt = """You help field marketing teams prep for conferences by answering questions on companies that they need to prepare to talk to. You produce quick and actionable briefs, doing your best to answer the user's question.

Today's date is {todays_date}.

You maintain a wiki — a knowledge base of interlinked markdown pages — instead of a frozen vector store. The wiki is your long-term memory for internal notes and anything else worth keeping.

Tools:
- ThinkTool: Plan and reason before you act. Use this when you need to think.
- DuckDuckGoSearchTool: Collect current information on agendas, speakers, news, competitor moves. Include title + date + link to the resources you find in your answer. Do not use this tool for internal notes.
- wikipedia_tool: Company/org history (not breaking news). Only look up company names as the input.
- wiki_list / wiki_search / wiki_read: Find and read pages in the internal wiki. Try wiki_search first.
- wiki_write / wiki_edit / wiki_link: Curate the wiki. When you ingest a document, decompose it into one page per distinct concept (entity, topic, product, event). Reference related pages inline with [[Other Page]] and record explicit links with wiki_link.

If you use information from the wiki in your response, label it [Internal]. Always consult the wiki when internal notes are referenced.

Basic Rules:
- Be concise and practical.
- Favor recent info (agenda/news ≤30 days; exec changes/funding ≤180 days); flag older items.
- If you don't know, say so. Don't make things up.
"""

In [ ]:
todays_date = date.today().strftime("%B %d, %Y")
formatted_instruct_prompt = instruct_prompt.format(todays_date=todays_date)

Memory:

In [ ]:
memory = SummarizeMemory(llm)

### Tools

In [ ]:
think_tool = ThinkTool()

The **DuckDuckGoSearchTool** is a Web Search tool that provides relevant data from the internet to the LLM.

In [ ]:
search_tool = DuckDuckGoSearchTool()

In [ ]:
@tool
async def wikipedia_tool(query: str) -> str:
    """Search Wikipedia for factual and historical information.

    Args:
        query: The topic or company name to look up.

    Returns:
        Text content from the matching Wikipedia article.
    """
    wiki = WikipediaTool(language="en")
    response = await wiki.run(WikipediaToolInput(query=query, full_text=False))
    return response.get_text_content()

### Wiki (replaces RAG)

A `WikiStore` holds `title -> markdown body` plus a directed link set. The agent mutates it through `wiki_*` tools, the same way the RAG version mutates nothing and only reads from a frozen vector store.

In [ ]:
class WikiStore:
    def __init__(self):
        self.pages: dict[str, str] = {}
        self.links: set[tuple[str, str]] = set()

    def list_pages(self):
        return sorted(self.pages.keys())

    def read(self, title):
        return self.pages.get(title)

    def write(self, title, content):
        self.pages[title] = content

    def edit(self, title, find, replace):
        if title not in self.pages:
            return "not found"
        if find not in self.pages[title]:
            return "no match"
        self.pages[title] = self.pages[title].replace(find, replace)
        return "ok"

    def add_link(self, source, target):
        self.links.add((source, target))

    def search(self, query, max_results=10):
        q = query.lower()
        hits = []
        for title, body in self.pages.items():
            if q in title.lower() or q in body.lower():
                hits.append((title, body[:200].replace("\n", " ")))
            if len(hits) >= max_results:
                break
        return hits

    def parsed_links(self):
        out = set(self.links)
        for title, body in self.pages.items():
            for target in re.findall(r"\[\[([^\]]+)\]\]", body):
                out.add((title, target.strip()))
        return out

    def save(self, path):
        with open(path, "w") as f:
            json.dump({"pages": self.pages, "links": sorted(self.parsed_links())}, f, indent=2)


STORE = WikiStore()

In [ ]:
@tool
async def wiki_list() -> str:
    """List the titles of every page currently in the internal wiki."""
    titles = STORE.list_pages()
    return "\n".join(titles) if titles else "(wiki is empty)"


@tool
async def wiki_search(query: str) -> str:
    """Search the internal wiki by case-insensitive substring (title or body).

    Args:
        query: Search query.

    Returns:
        Titles and short snippets of matching pages.
    """
    hits = STORE.search(query)
    if not hits:
        return f"(no pages matched '{query}')"
    return "\n".join(f"- {t}: {s}" for t, s in hits)


@tool
async def wiki_read(title: str) -> str:
    """Read the full markdown body of a single wiki page.

    Args:
        title: Exact page title.
    """
    body = STORE.read(title)
    return body if body is not None else f"(page '{title}' not found)"


@tool
async def wiki_write(title: str, content: str) -> str:
    """Create a new wiki page or overwrite an existing one.

    Args:
        title: Short noun-phrase title (e.g. "Moderna", "mRNA Therapies").
        content: Markdown body. Use [[Other Page]] to reference other pages inline.
    """
    STORE.write(title, content)
    return f"ok (wrote '{title}', {len(content)} chars)"


@tool
async def wiki_edit(title: str, find: str, replace: str) -> str:
    """Patch a page by replacing one exact substring with another.

    Args:
        title: Page to edit.
        find: Exact substring to find.
        replace: Replacement text.
    """
    return STORE.edit(title, find, replace)


@tool
async def wiki_link(source: str, target: str) -> str:
    """Record an explicit directed link from one page to another.

    Args:
        source: Title the link starts from.
        target: Title the link points to.
    """
    STORE.add_link(source, target)
    return "ok"

### Seed the Wiki from Internal Notes

We read the same synthetic internal-notes file the RAG version used, hand it to the agent once, and ask it to decompose it into wiki pages. After this step, `STORE` is the agent's persistent memory of internal context — no vectors, no embeddings.

In [ ]:
with open("/content/beeai_framework_synthetic_data/rag_conference_prep_agent.txt") as f:
    internal_notes = f.read()
print(f"Loaded {len(internal_notes)} chars of internal notes")

### Control Agent Behavior

In [ ]:
requirement_1 = ConditionalRequirement(ThinkTool, consecutive_allowed=False, force_at_step=1)

requirement_2 = ConditionalRequirement(wikipedia_tool, only_after=ThinkTool, consecutive_allowed=True, priority=10)

requirement_3 = ConditionalRequirement(DuckDuckGoSearchTool, only_after=ThinkTool, consecutive_allowed=True, min_invocations=1, max_invocations=3, priority=15)

requirement_4 = ConditionalRequirement(wiki_search, only_after=ThinkTool, consecutive_allowed=True, priority=18)
requirement_5 = ConditionalRequirement(wiki_read, only_after=ThinkTool, consecutive_allowed=True, priority=16)
requirement_6 = ConditionalRequirement(wiki_list, only_after=ThinkTool, consecutive_allowed=False, priority=14)
requirement_7 = ConditionalRequirement(wiki_write, only_after=ThinkTool, consecutive_allowed=True, priority=12)
requirement_8 = ConditionalRequirement(wiki_edit, only_after=ThinkTool, consecutive_allowed=True, priority=11)
requirement_9 = ConditionalRequirement(wiki_link, only_after=ThinkTool, consecutive_allowed=True, priority=8)

In [ ]:
def setup_observability(endpoint: str = "http://localhost:6006/v1/traces"):
    tracer_provider = register(project_name="conference-prep-agent-wiki", endpoint=endpoint)
    BeeAIInstrumentor().instrument(tracer_provider=tracer_provider)

In [ ]:
load_dotenv()
px_session = px.launch_app()
setup_observability("http://localhost:6006/v1/traces")
print(f"Phoenix UI: {px_session.url}")

## Assemble Agent

In [ ]:
agent = RequirementAgent(
    llm=llm,
    instructions=formatted_instruct_prompt,
    memory=memory,
    tools=[
        ThinkTool(),
        DuckDuckGoSearchTool(),
        wikipedia_tool,
        wiki_list,
        wiki_search,
        wiki_read,
        wiki_write,
        wiki_edit,
        wiki_link,
    ],
    requirements=[
        requirement_1, requirement_2, requirement_3,
        requirement_4, requirement_5, requirement_6,
        requirement_7, requirement_8, requirement_9,
    ],
    middlewares=[GlobalTrajectoryMiddleware(included=[Tool])],
)

### Ingest internal notes into the wiki (one-time)

In [ ]:
ingest = await agent.run(
    f"Read the following internal notes and build a wiki from them. Create one page per distinct concept (company, person, product, event) and connect related pages with [[wikilinks]] and explicit wiki_link calls. Keep pages concise.\n\n---\n{internal_notes}\n---",
    max_retries_per_step=3,
    total_max_retries=25,
)
print(ingest.last_message.text)

### Inspect the wiki the agent built

In [ ]:
print("=" * 60)
print(f"PAGES ({len(STORE.pages)})")
print("=" * 60)
for title in STORE.list_pages():
    print(f"\n## {title}")
    print(STORE.read(title))

print("\n" + "=" * 60)
print(f"LINKS ({len(STORE.parsed_links())})")
print("=" * 60)
for src, dst in sorted(STORE.parsed_links()):
    print(f"  {src} -> {dst}")

In [ ]:
question = "I'm planning on meeting the Moderna rep at the next conference. Give me a one pager and remind me where we left off on previous internal discussions."

### Run Agent

In [ ]:
response = await agent.run(question, max_retries_per_step=3, total_max_retries=25)
print(response.last_message.text)

In [ ]:
px_session.view()

### Persist the wiki

In [ ]:
STORE.save("wiki_store.json")
print(f"Saved {len(STORE.pages)} pages and {len(STORE.parsed_links())} links to wiki_store.json")